---

## 5. Modelling

### 5.1 Membangun Algoritma

In [ ]:
import numpy as np

class LightGBM:
    def __init__(self, n_estimators=100, learning_rate=0.05, max_depth=5, num_leaves=15,
                 min_child_samples=10, max_bins=64, lambda_reg=1.0, gamma=0.0, scale_pos_weight=1.0):
        self.n_estimators, self.learning_rate, self.max_depth = n_estimators, learning_rate, max_depth
        self.num_leaves, self.min_child_samples, self.max_bins = num_leaves, min_child_samples, max_bins
        self.lambda_reg, self.gamma, self.scale_pos_weight = lambda_reg, gamma, scale_pos_weight

    def _sigmoid(self, x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def _make_bins(self, X):
        X_bin, self.thresholds, q = np.zeros(X.shape, dtype=int), [], np.linspace(0, 1, self.max_bins + 1)[1:-1]
        for j in range(X.shape[1]):
            vals = np.unique(X[:, j])
            cuts = (vals[:-1] + vals[1:]) / 2 if len(vals) <= self.max_bins else np.unique(np.quantile(X[:, j], q))
            self.thresholds.append(cuts)
            X_bin[:, j] = np.searchsorted(cuts, X[:, j])
        return X_bin

    def _best_split(self, idx, X_bin, g, h):
        if len(idx) < 2 * self.min_child_samples: return None
        G, H, best = g[idx].sum(), h[idx].sum(), (0.0, None, None, None)
        for j, cuts in enumerate(self.thresholds):
            if len(cuts) == 0: continue
            bins, size = X_bin[idx, j], len(cuts) + 1
            g_l = np.cumsum(np.bincount(bins, weights=g[idx], minlength=size))[:-1]
            h_l = np.cumsum(np.bincount(bins, weights=h[idx], minlength=size))[:-1]
            c_l = np.cumsum(np.bincount(bins, minlength=size))[:-1]
            valid = (c_l >= self.min_child_samples) & (len(idx) - c_l >= self.min_child_samples)
            if not valid.any(): continue
            gain = 0.5 * (g_l**2 / (h_l + self.lambda_reg) + (G - g_l)**2 / (H - h_l + self.lambda_reg) - G**2 / (H + self.lambda_reg)) - self.gamma
            k = np.argmax(np.where(valid, gain, -np.inf))
            if gain[k] > best[0]: best = (gain[k], j, k, cuts[k])
        return best if best[1] is not None else None

    def _build_tree(self, X_bin, g, h):
        idx, leaf_val = np.arange(len(g)), lambda i: -g[i].sum() / (h[i].sum() + self.lambda_reg)
        root, split = {"value": leaf_val(idx), "depth": 0}, self._best_split(idx, X_bin, g, h)
        cands, leaves = ([(split, root, idx)] if split else []), 1
        while cands and leaves < self.num_leaves:
            (_, feat, cut_bin, thresh), node, n_idx = cands.pop(np.argmax([c[0][0] for c in cands]))
            mask = X_bin[n_idx, feat] <= cut_bin
            node["feature"], node["threshold"] = feat, thresh
            for side, c_idx in (("left", n_idx[mask]), ("right", n_idx[~mask])):
                node[side] = {"value": leaf_val(c_idx), "depth": node["depth"] + 1}
                s = self._best_split(c_idx, X_bin, g, h) if node[side]["depth"] < self.max_depth else None
                if s: cands.append((s, node[side], c_idx))
            leaves += 1
        return root

    def _predict_tree(self, node, X):
        if "feature" not in node: return np.full(len(X), node["value"])
        mask, out = X[:, node["feature"]] <= node["threshold"], np.zeros(len(X))
        if mask.any(): out[mask] = self._predict_tree(node["left"], X[mask])
        if (~mask).any(): out[~mask] = self._predict_tree(node["right"], X[~mask])
        return out

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y, dtype=float)
        w, X_bin = np.where(y == 1, self.scale_pos_weight, 1.0), self._make_bins(X)
        rate = np.clip(np.sum(w * y) / np.sum(w), 1e-15, 1 - 1e-15)
        self.base_score, self.trees = np.log(rate / (1 - rate)), []
        raw = np.full(len(y), self.base_score)
        for _ in range(self.n_estimators):
            p = self._sigmoid(raw)
            tree = self._build_tree(X_bin, w * (p - y), np.maximum(w * p * (1 - p), 1e-16))
            self.trees.append(tree)
            raw += self.learning_rate * self._predict_tree(tree, X)
        return self

    def predict_prob(self, X):
        X = np.asarray(X, dtype=float)
        return self._sigmoid(self.base_score + sum((self.learning_rate * self._predict_tree(t, X) for t in self.trees), np.zeros(len(X))))
    def predict(self, X, threshold=0.5): return (self.predict_prob(X) >= threshold).astype(int)

### 5.2 Pembagian Data dan Pelatihan Model

In [ ]:
def stratified_kfold(y, k=5, seed=42):
    np.random.seed(seed)
    folds = [[] for _ in range(k)]
    for label in np.unique(y):
        idx = np.where(y == label)[0]
        np.random.shuffle(idx)
        for i, part in enumerate(np.array_split(idx, k)):
            folds[i].extend(part)
    return [
        (np.concatenate([folds[j] for j in range(k) if j != i]), np.array(folds[i]))
        for i in range(k)
    ]

pos_weight = np.sum(y == 0) / np.sum(y == 1)
folds = stratified_kfold(y, k=5)
models, results = [], []

for i, (train_idx, val_idx) in enumerate(folds, 1):
    model = LightGBM(scale_pos_weight=pos_weight).fit(X[train_idx], y[train_idx])
    y_val = y[val_idx]
    y_prob = model.predict_prob(X[val_idx])
    y_pred = model.predict(X[val_idx])
    models.append(model)
    results.append((val_idx, y_val, y_prob, y_pred))
    print(f"Fold {i} | Train: {len(train_idx)} | Val: {len(val_idx)} | Pos Val: {int(y_val.sum())}")

### 5.3 Hyperparameter Tuning

In [ ]:
import matplotlib.pyplot as plt

def log_loss(y_true, y_prob, pos_w=1.0):
    p = np.clip(y_prob, 1e-15, 1 - 1e-15)
    w = np.where(y_true == 1, pos_w, 1.0)
    return -np.sum(w * (y_true * np.log(p) + (1 - y_true) * np.log(1 - p))) / np.sum(w)

def count_splits(trees, n_features):
    counts, stack = np.zeros(n_features, dtype=int), list(trees)
    while stack:
        node = stack.pop()
        if "feature" in node:
            counts[node["feature"]] += 1
            stack.extend((node["left"], node["right"]))
    return counts

param_grid = [
    {"n_estimators": 50,  "learning_rate": 0.05, "num_leaves": 7,  "max_depth": 3, "lambda_reg": 2.0},
    {"n_estimators": 80,  "learning_rate": 0.05, "num_leaves": 10, "max_depth": 4, "lambda_reg": 2.0},
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 15, "max_depth": 5, "lambda_reg": 1.0},
    {"n_estimators": 120, "learning_rate": 0.03, "num_leaves": 10, "max_depth": 4, "lambda_reg": 3.0},
]

best_loss, best_params, best_models, best_results = np.inf, None, None, None
for i, params in enumerate(param_grid, 1):
    f_models, f_results, losses = [], [], []
    for train_idx, val_idx in folds:
        m = LightGBM(**params, scale_pos_weight=pos_weight).fit(X[train_idx], y[train_idx])
        prob = m.predict_prob(X[val_idx])
        f_models.append(m)
        f_results.append((val_idx, y[val_idx], prob, (prob >= 0.5).astype(int)))
        losses.append(log_loss(y[val_idx], prob, pos_weight))
    mean_loss = np.mean(losses)
    print(f"Config {i} | Val Log Loss: {mean_loss:.4f} | {params}")
    if mean_loss < best_loss:
        best_loss, best_params, best_models, best_results = mean_loss, params, f_models, f_results

models, results = best_models, best_results
oof_prob, oof_pred = np.zeros(len(y)), np.zeros(len(y), dtype=int)
for val_idx, _, prob, pred in results:
    oof_prob[val_idx], oof_pred[val_idx] = prob, pred

final_model = LightGBM(**best_params, scale_pos_weight=pos_weight).fit(X, y)
importances = count_splits(final_model.trees, X.shape[1])
order = np.argsort(importances)
names = np.array(feature_names if "feature_names" in globals() else [f"Fitur {i+1}" for i in range(X.shape[1])])
plt.barh(names[order], importances[order], color="steelblue")
plt.title(f"Feature Importance LightGBM (Best Log Loss: {best_loss:.4f})")
plt.xlabel("Jumlah Split")
plt.show()

---

## 6. Evaluation

### 6.1 Evaluasi Performa Model

In [ ]:
import pandas as pd
import seaborn as sns

def calc_roc_auc(y_true, y_prob):
    thresholds = np.r_[np.inf, np.sort(np.unique(y_prob))[::-1]]
    pos, neg = max(np.sum(y_true == 1), 1), max(np.sum(y_true == 0), 1)
    tpr = np.array([np.sum((y_prob >= t) & (y_true == 1)) / pos for t in thresholds])
    fpr = np.array([np.sum((y_prob >= t) & (y_true == 0)) / neg for t in thresholds])
    auc = np.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) / 2)
    return fpr, tpr, auc

def evaluate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    acc = (tp + tn) / len(y_true)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    denom = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0
    _, _, auc = calc_roc_auc(y_true, y_prob)
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "ROC-AUC": auc, "MCC": mcc}

rows = [{"Fold": f"Fold {i}", **evaluate_metrics(yv, yp)} for i, (_, yv, yp, _) in enumerate(results, 1)]
eval_df = pd.DataFrame(rows)
mean_row = {"Fold": "Rata-rata", **eval_df.drop(columns="Fold").mean().to_dict()}
oof_metrics = evaluate_metrics(y, oof_prob)
summary_df = pd.concat([eval_df, pd.DataFrame([mean_row, {"Fold": "OOF Total", **oof_metrics}])], ignore_index=True)
print(summary_df.round(4).to_string(index=False))

cm = np.array([
    [np.sum((y == 0) & (oof_pred == 0)), np.sum((y == 0) & (oof_pred == 1))],
    [np.sum((y == 1) & (oof_pred == 0)), np.sum((y == 1) & (oof_pred == 1))]
])
fpr, tpr, oof_auc = calc_roc_auc(y, oof_prob)
labels = ["Tidak (0)", "Kecelakaan (1)"]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, ax=ax[0])
ax[0].set(title="Confusion Matrix (OOF)", xlabel="Prediksi", ylabel="Aktual")
ax[1].plot(fpr, tpr, color="navy", lw=2, label=f"ROC (AUC = {oof_auc:.4f})")
ax[1].plot([0, 1], [0, 1], "r--", lw=1)
ax[1].set(title="Kurva ROC-AUC (OOF)", xlabel="False Positive Rate", ylabel="True Positive Rate")
ax[1].grid(True, linestyle="--", alpha=0.5)
ax[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

### 6.2 Noise Pertubation Testing

In [ ]:
def noise_perturbation_test(models, folds, X, y, levels=(0.0, 0.05, 0.10, 0.20), seed=42):
    rng, feat_std, rows = np.random.default_rng(seed), np.std(X, axis=0), []
    for lvl in levels:
        prob_oof = np.zeros(len(y))
        for model, (_, val_idx) in zip(models, folds):
            X_val = X[val_idx].astype(float)
            X_noisy = X_val + rng.normal(0, lvl * feat_std, X_val.shape) if lvl > 0 else X_val
            prob_oof[val_idx] = model.predict_prob(X_noisy)
        rows.append({"Noise": f"{int(lvl * 100)}%", **evaluate_metrics(y, prob_oof)})
    return pd.DataFrame(rows)

noise_df = noise_perturbation_test(models, folds, X, y)
noise_df["Drop_Acc"] = noise_df["Accuracy"].iloc[0] - noise_df["Accuracy"]
print(noise_df.round(4).to_string(index=False))

plt.figure(figsize=(8, 4))
for col in ["Accuracy", "Recall", "F1-Score", "ROC-AUC"]:
    plt.plot(noise_df["Noise"], noise_df[col], marker="o", lw=2, label=col)
plt.title("Noise Perturbation Testing — Ketahanan Model LightGBM")
plt.xlabel("Tingkat Noise")
plt.ylabel("Skor Metrik")
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()